# AireIQ · S2: entorno y lectura de humedad
**Responsable:** Eberth Emmanuel Rosales Castillo.

**Descriptiva:** ¿Cómo varía la humedad según la hora y el día y qué porcentaje de lecturas válidas queda fuera del rango histórico p5–p95?

**Predictiva U1:** ¿Qué tan precisamente se estima la humedad con temperatura, presión y hora?

**Predictiva U2:** ¿Cuál será la humedad dentro de un minuto y saldrá del rango histórico?

Este cuaderno comprueba el entorno y la fuente. S3 añadirá calidad completa y Parquet; S4, modelos. El rango histórico no es un umbral sanitario. Arquitectura Lambda: Supabase alimentará batch y streaming. El CSV original se monta como solo lectura.

Referencia de entorno: https://github.com/262bigdata/lambda26/tree/main/pyspark


In [1]:
import sys
from pathlib import Path
import pyspark
from pyspark.sql import SparkSession, functions as F, types as T

spark = (SparkSession.builder.appName("AireIQ-S2-Humedad")
         .master("local[2]")
         .config("spark.sql.session.timeZone", "America/Lima")
         .config("spark.sql.shuffle.partitions", "4")
         .getOrCreate())
print("Python:", sys.version)
print("PySpark:", pyspark.__version__)
print("Master:", spark.sparkContext.master)
print("Modo de laboratorio: dos hilos locales; no es un cluster multinodo.")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/20 02:38:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Python: 3.10.22 (main, Oct  6 2026, 02:05:33) [GCC 14.2.0]
PySpark: 4.2.0
Master: local[2]
Modo de laboratorio: dos hilos locales; no es un cluster multinodo.


In [2]:
path = Path("/opt/aireiq/data/mediciones_aire_rows.csv")
assert path.is_file(), "Falta el CSV montado por Compose"
schema = T.StructType([
    T.StructField("id", T.StringType()),
    T.StructField("created_at", T.TimestampType()),
    T.StructField("dispositivo", T.StringType()),
    *[T.StructField(c, T.DoubleType()) for c in [
        "temperatura_c", "humedad_pct", "presion_hpa", "gas_ohms", "altura_m",
        "iaq_estimado", "eco2_estimado_ppm", "voc_indice_estimado"]],
    T.StructField("calidad_aire", T.StringType())
])
df = (spark.read.option("header", True).option("mode", "FAILFAST")
      .option("enforceSchema", False).schema(schema).csv(str(path)))
df.printSchema()
df.select("created_at", "dispositivo", "humedad_pct", "temperatura_c", "presion_hpa").show(5, truncate=False)


root
 |-- id: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- dispositivo: string (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- humedad_pct: double (nullable = true)
 |-- presion_hpa: double (nullable = true)
 |-- gas_ohms: double (nullable = true)
 |-- altura_m: double (nullable = true)
 |-- iaq_estimado: double (nullable = true)
 |-- eco2_estimado_ppm: double (nullable = true)
 |-- voc_indice_estimado: double (nullable = true)
 |-- calidad_aire: string (nullable = true)

+--------------------------+------------+-----------+-------------+-----------+
|created_at                |dispositivo |humedad_pct|temperatura_c|presion_hpa|
+--------------------------+------------+-----------+-------------+-----------+
|2026-10-21 23:52:48.394234|ESP32_AireIQ|37.3878    |18.7746      |647.35     |
|2026-10-21 23:53:50.091276|ESP32_AireIQ|37.3368    |18.7791      |647.34     |
|2026-10-21 23:54:51.627012|ESP32_AireIQ|37.251     |18.8374      |647.33

## Transformación y evaluación perezosa
Definir columnas y filtros construye un plan. `explain()` permite inspeccionarlo; `count()` y `show()` solicitan resultados. El filtro siguiente aplica la regla inicial del brief, no sustituye la validación completa de S3. No se eliminan registros del CSV.

In [3]:
validas = df.filter((F.col("gas_ohms") > 0) & F.col("humedad_pct").between(0, 100))
por_hora = (validas.withColumn("hora_lima", F.hour("created_at"))
            .groupBy("hora_lima")
            .agg(F.count("*").alias("lecturas"), F.avg("humedad_pct").alias("humedad_promedio_pct")))
por_hora.explain(True)
total = df.count()
conservadas = validas.count()
print({"total": total, "conservadas_regla_inicial": conservadas, "excluidas": total-conservadas})
assert total > 0 and 0 < conservadas <= total
assert df.filter(F.col("created_at").isNull()).count() == 0, "Hay fechas no interpretadas"
por_hora.orderBy("hora_lima").show(24)


== Parsed Logical Plan ==
'Aggregate ['hora_lima], ['hora_lima, 'count(*) AS lecturas#35, 'avg('humedad_pct) AS humedad_promedio_pct#36]
+- Project [id#0, created_at#1, dispositivo#2, temperatura_c#3, humedad_pct#4, presion_hpa#5, gas_ohms#6, altura_m#7, iaq_estimado#8, eco2_estimado_ppm#9, voc_indice_estimado#10, calidad_aire#11, hour(created_at#1, Some(America/Lima)) AS hora_lima#34]
   +- Filter ((gas_ohms#6 > cast(0 as double)) AND ((humedad_pct#4 >= cast(0 as double)) AND (humedad_pct#4 <= cast(100 as double))))
      +- Relation [id#0,created_at#1,dispositivo#2,temperatura_c#3,humedad_pct#4,presion_hpa#5,gas_ohms#6,altura_m#7,iaq_estimado#8,eco2_estimado_ppm#9,voc_indice_estimado#10,calidad_aire#11] csv

== Analyzed Logical Plan ==
hora_lima: int, lecturas: bigint, humedad_promedio_pct: double
Aggregate [hora_lima#34], [hora_lima#34, count(1) AS lecturas#35L, avg(humedad_pct#4) AS humedad_promedio_pct#36]
+- Project [id#0, created_at#1, dispositivo#2, temperatura_c#3, humedad_pct

In [4]:
# Ejemplo RDD requerido en S2: conteo por dispositivo.
conteos_rdd = (validas.select("dispositivo").rdd
               .map(lambda row: (row.dispositivo, 1))
               .reduceByKey(lambda a, b: a + b).collect())
print(conteos_rdd)
assert sum(n for _, n in conteos_rdd) == conservadas


[Stage 13:>                                                         (0 + 1) / 1]

[('ESP32_AireIQ', 12853)]


## Evidencia
Ejecutar todas las celdas y guardar el cuaderno ejecutado. No se han precargado salidas.
El promedio mostrado pondera lecturas, no minutos; S3 tratará el cambio de frecuencia de captura.
Siguiente etapa: nulos, duplicados, rangos, cuarentena y Parquet particionado por fecha, con lectura de vuelta.
